In [2]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# ------------------ CONFIG ------------------

benchmarks = [
    "private_enterprise",
    "social_media_cloud",
    "commercial_cloud",
    "university"
]

#benchmarks = [
#    "private_enterprise",
#    "social_media_cloud"
#]

loads = range(1, 10)

input_dir = "/home/hsd/workspace/trafpy/examples/comparison_generator/final_data"

# Existing ns3 outputs
output_dir = "/home/hsd/workspace/ns3-load-balance/results_fix"

# RTT outputs (change this)
rtt_dir = "/home/hsd/workspace/ns3-load-balance/results_rtt_fix"

save_dir = "/home/hsd/workspace/ns3-load-balance/results_rtt/analysis_results"
Path(save_dir).mkdir(exist_ok=True)

ip_pattern = r'^\d+\.\d+\.\d+\.\d+$'

summary_results = {}

# ------------------ FUNCTIONS ------------------

def clean(df):
    return df[
        df['Src'].str.match(ip_pattern, na=False) &
        df['Dest'].str.match(ip_pattern, na=False)
    ].copy()


def ip_to_node(ip):
    last = int(ip.split('.')[-1])
    last1 = int(ip.split('.')[-2])

    leafcount = 2
    leaf = 0

    if last1 == 2:
        leaf = 1

    return ((last // 2) - 1 + (leaf * leafcount))


def add_time(df):

    df['start_time'] = (
        df['TimeFirstTxPacket']
        .astype(str)
        .str.replace('+', '', regex=False)
        .str.replace('ns', '', regex=False)
        .astype(float) / 1e9
    )

    return df


flow_columns = [
    "FlowID",
    "Src",
    "Dest",
    "TimeFirstRxPacket",
    "TimeFirstTxPacket",
    "TimeLastRxPacket",
    "TimeLastTxPacket",
    "FCT(s)",
    "TxPackets",
    "RxPackets",
    "LostPackets",
    "LossRate",
    "PDR",
    "LossPercent",
    "TxBytes",
    "RxBytes",
    "Throughput(Kbps)",
    "MeanDelay(ms)",
    "Jitter(ms)",
    "HopCount"
]


# ------------------ MATCH ------------------

def match_df(output_df, input_df, label):

    matches = []

    for _, in_row in input_df.iterrows():

        cand = output_df[
            (output_df.sn == in_row.sn) &
            (output_df.dn == in_row.dn)
        ].copy()

        if len(cand) == 0:
            continue

        cand['time_diff'] = abs(
            cand['start_time'] - in_row.event_time
        )

        best = cand.loc[cand['time_diff'].idxmin()]

        combined = {
            'flow_id': in_row.flow_id,
            'sn': in_row.sn,
            'dn': in_row.dn,
            'input_time': in_row.event_time,
            'flow_size': in_row.flow_size,

            f'time_diff_{label}': best['time_diff'],

            f'TxPackets_{label}': best['TxPackets'],
            f'RxPackets_{label}': best['RxPackets'],

            f'TxBytes_calc_{label}': best['TxPackets'] * 1400,
            f'RxBytes_calc_{label}': best['RxPackets'] * 1400,
        }

        for col in flow_columns:
            combined[f"{col}_{label}"] = best[col]

        matches.append(combined)

    return pd.DataFrame(matches)


# ------------------ MAIN ------------------

for benchmark in benchmarks:

    for load in loads:

        print(f"\n=== {benchmark} | Load 0.{load} ===")

        input_path = (
            f"{input_dir}/{benchmark}_load_{load}.csv"
        )

        try:

            input_df = pd.read_csv(input_path)

            algorithms = {
                "conga": pd.read_csv(
                    f"{output_dir}/{benchmark}_load_{load}_Conga.csv",
                    nrows=12000
                ),

                "ecmp": pd.read_csv(
                    f"{output_dir}/{benchmark}_load_{load}_ECMP.csv",
                    nrows=12000
                ),

                "rtt": pd.read_csv(
                    f"{rtt_dir}/{benchmark}_load_{load}_RTT.csv",
                    nrows=12000
                )
            }

        except Exception as e:
            print("Skipping:", e)
            continue

        # ------------------ CLEAN ------------------

        for label, df in algorithms.items():

            df = add_time(clean(df))

            df['sn'] = df['Src'].apply(ip_to_node)
            df['dn'] = df['Dest'].apply(ip_to_node)

            algorithms[label] = df

        # ------------------ MATCH ------------------

        matched = {}

        for label, df in algorithms.items():
            matched[label] = match_df(df, input_df, label)

        merge_keys = [
            'flow_id',
            'sn',
            'dn',
            'input_time',
            'flow_size'
        ]

        final_df = matched['conga']

        for algo in ['ecmp', 'rtt']:

            final_df = pd.merge(
                final_df,
                matched[algo],
                on=merge_keys,
                how='inner'
            )

        if len(final_df) == 0:
            print("No matches.")
            continue

        # ------------------ DIFFERENCES ------------------

        final_df['fct_diff_conga_ecmp'] = (
            final_df['FCT(s)_conga']
            - final_df['FCT(s)_ecmp']
        )

        final_df['fct_diff_conga_rtt'] = (
            final_df['FCT(s)_conga']
            - final_df['FCT(s)_rtt']
        )

        final_df['fct_diff_ecmp_rtt'] = (
            final_df['FCT(s)_ecmp']
            - final_df['FCT(s)_rtt']
        )

        # ------------------ SAVE CSV ------------------

        csv_path = (
            f"{save_dir}/{benchmark}_load_{load}_full.csv"
        )

        final_df.to_csv(csv_path, index=False)

        # ------------------ STATS ------------------

        avg_conga = final_df['FCT(s)_conga'].mean()
        avg_ecmp = final_df['FCT(s)_ecmp'].mean()
        avg_rtt = final_df['FCT(s)_rtt'].mean()

        # Choose percentile
        PERCENTILE = 90      # change to 75, 99, etc.


        per_conga = final_df['FCT(s)_conga'].quantile(PERCENTILE / 100.0)
        per_ecmp = final_df['FCT(s)_ecmp'].quantile(PERCENTILE / 100.0)
        per_rtt = final_df['FCT(s)_rtt'].quantile(PERCENTILE / 100.0)

        print("Flows:", len(final_df))
        print("Avg Conga:", avg_conga)
        print("Avg ECMP :", avg_ecmp)
        print("Avg RTT  :", avg_rtt)
        print("Per Conga:", per_conga)
        print("Per ECMP :", per_ecmp)
        print("Per RTT  :", per_rtt)

        if benchmark not in summary_results:

            summary_results[benchmark] = {
                'loads': [],
                'conga': [],
                'ecmp': [],
                'rtt': []
            }

        summary_results[benchmark]['loads'].append(load / 10)
        summary_results[benchmark]['conga'].append(avg_conga)
        summary_results[benchmark]['ecmp'].append(avg_ecmp)
        summary_results[benchmark]['rtt'].append(avg_rtt)

        # ------------------ SCATTERS ------------------

        pairs = [
            ('conga', 'ecmp'),
            ('conga', 'rtt'),
            ('ecmp', 'rtt')
        ]

        for a, b in pairs:

            max_val = max(
                final_df[f'FCT(s)_{a}'].max(),
                final_df[f'FCT(s)_{b}'].max()
            )

            plt.figure(figsize=(6, 6))

            plt.scatter(
                final_df[f'FCT(s)_{a}'],
                final_df[f'FCT(s)_{b}'],
                alpha=0.5
            )

            plt.plot(
                [0, max_val],
                [0, max_val],
                '--'
            )

            plt.xlabel(f"{a.upper()} FCT")
            plt.ylabel(f"{b.upper()} FCT")

            plt.title(
                f"{benchmark} Load 0.{load}"
            )

            plt.grid()

            plt.savefig(
                f"{save_dir}/{benchmark}_load_{load}_{a}_{b}_scatter.png"
            )

            plt.close()

        # ------------------ HISTOGRAMS ------------------

        histograms = [
            ('fct_diff_conga_ecmp', 'conga_ecmp'),
            ('fct_diff_conga_rtt', 'conga_rtt'),
            ('fct_diff_ecmp_rtt', 'ecmp_rtt')
        ]

        for col, name in histograms:

            plt.figure()

            plt.hist(
                final_df[col],
                bins=30
            )

            plt.xlabel(col)
            plt.ylabel("Count")

            plt.title(
                f"{benchmark} Load 0.{load}"
            )

            plt.grid()

            plt.savefig(
                f"{save_dir}/{benchmark}_load_{load}_{name}_hist.png"
            )

            plt.close()

# ------------------ FINAL SUMMARY PLOTS ------------------

for benchmark in summary_results:

    loads_x = summary_results[benchmark]['loads']

    conga_avg = summary_results[benchmark]['conga']
    ecmp_avg = summary_results[benchmark]['ecmp']
    rtt_avg = summary_results[benchmark]['rtt']

    plt.figure(figsize=(8, 5))

    plt.plot(
        loads_x,
        conga_avg,
        marker='o',
        label='Conga'
    )

    plt.plot(
        loads_x,
        ecmp_avg,
        marker='s',
        label='ECMP'
    )

    plt.plot(
        loads_x,
        rtt_avg,
        marker='^',
        label='RTT'
    )

    plt.xlabel("Load")
    plt.ylabel("Average FCT")
    plt.title(
        f"{benchmark} Avg FCT vs Load"
    )

    plt.legend()
    plt.grid()

    plt.savefig(
        f"{save_dir}/{benchmark}_avg_fct_vs_load.png"
    )

    plt.close()

print("\nDONE: Conga vs ECMP vs RTT comparison completed!")


=== private_enterprise | Load 0.1 ===
Flows: 6000
Avg Conga: 0.0064150025
Avg ECMP : 0.0064381625
Avg RTT  : 0.006433049833333333
Per Conga: 0.01875840000000001
Per ECMP : 0.01884110000000001
Per RTT  : 0.018461700000000025

=== private_enterprise | Load 0.2 ===
Flows: 6000
Avg Conga: 0.009239524499999999
Avg ECMP : 0.009333161833333332
Avg RTT  : 0.009264714166666667
Per Conga: 0.02578770000000001
Per ECMP : 0.026381200000000053
Per RTT  : 0.025354500000000012

=== private_enterprise | Load 0.3 ===
Flows: 6000
Avg Conga: 0.012709421833333333
Avg ECMP : 0.012475241000000001
Avg RTT  : 0.012912018666666669
Per Conga: 0.033743100000000005
Per ECMP : 0.03336580000000002
Per RTT  : 0.03364770000000004

=== private_enterprise | Load 0.4 ===
Flows: 6000
Avg Conga: 0.013811163833333334
Avg ECMP : 0.013959376166666667
Avg RTT  : 0.0137935995
Per Conga: 0.0359082
Per ECMP : 0.035905400000000004
Per RTT  : 0.03591100000000001

=== private_enterprise | Load 0.5 ===
Flows: 6000
Avg Conga: 0.01964

In [6]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import numpy as np

# ------------------ CONFIG ------------------

save_dir = Path(
    "/home/hsd/workspace/ns3-load-balance/results_rtt_fix/analysis_results/"
)

graph_dir = save_dir / "graph"
# Create the "graph" directory if it doesn't exist
graph_dir.parent.mkdir(parents=True, exist_ok=True)
graph_dir.mkdir(exist_ok=True)

benchmarks = [
    "private_enterprise",
    "social_media_cloud",
    "commercial_cloud",
    "university"
]

loads = range(1, 10)

SMALL = 100 * 1024        # 100 KB
LARGE = 1 * 1024 * 1024  # 1 MB

# ------------------ ANALYSIS ------------------

for benchmark in benchmarks:

    print(f"\n===== {benchmark} =====")

    load_vals = []

    # ---------- SMALL FLOWS ----------
    small_conga = []
    small_ecmp = []
    small_rtt = []

    # ---------- LARGE FLOWS ----------
    large_conga = []
    large_ecmp = []
    large_rtt = []

    for load in loads:

        csv_path = save_dir / f"{benchmark}_load_{load}_full.csv"

        # Create the "graph" directory if it doesn't exist
        csv_path.parent.mkdir(parents=True, exist_ok=True)

        if not csv_path.exists():
            print(f"Missing: {csv_path}")
            continue

        df = pd.read_csv(csv_path)

        if df.empty:
            continue

        # ------------------ SPLIT FLOWS ------------------

        small_df = df[df["flow_size"] < SMALL]
        large_df = df[df["flow_size"] > LARGE]

        # ------------------ SMALL FLOWS ------------------

        if not small_df.empty:

            small_conga.append(
                small_df["FCT(s)_conga"].mean()
            )

            small_ecmp.append(
                small_df["FCT(s)_ecmp"].mean()
            )

            small_rtt.append(
                small_df["FCT(s)_rtt"].mean()
            )

        else:

            small_conga.append(np.nan)
            small_ecmp.append(np.nan)
            small_rtt.append(np.nan)

        # ------------------ LARGE FLOWS ------------------

        if not large_df.empty:

            large_conga.append(
                large_df["FCT(s)_conga"].mean()
            )

            large_ecmp.append(
                large_df["FCT(s)_ecmp"].mean()
            )

            large_rtt.append(
                large_df["FCT(s)_rtt"].mean()
            )

        else:

            large_conga.append(np.nan)
            large_ecmp.append(np.nan)
            large_rtt.append(np.nan)

        load_vals.append(load / 10)

    # ==================================================
    # SMALL FLOWS
    # ==================================================

    plt.figure(figsize=(8, 5))

    plt.plot(
        load_vals,
        small_conga,
        marker='o',
        linewidth=2,
        label='Conga'
    )

    plt.plot(
        load_vals,
        small_ecmp,
        marker='s',
        linewidth=2,
        label='ECMP'
    )

    plt.plot(
        load_vals,
        small_rtt,
        marker='^',
        linewidth=2,
        label='RTT'
    )

    plt.xlabel("Network Load")
    plt.ylabel("Average FCT (s)")

    plt.title(
        f"{benchmark}: Small Flows (<100KB)"
    )

    plt.legend()
    plt.grid(True)

    out_path = (
        graph_dir /
        f"{benchmark}_SMALL_vs_load.png"
    )

    # Create the "graph" directory if it doesn't exist
    out_path.parent.mkdir(parents=True, exist_ok=True)

    plt.savefig(out_path, bbox_inches='tight')
    plt.close()

    print(f"Saved: {out_path}")

    # ==================================================
    # LARGE FLOWS
    # ==================================================

    plt.figure(figsize=(8, 5))

    plt.plot(
        load_vals,
        large_conga,
        marker='o',
        linewidth=2,
        label='Conga'
    )

    plt.plot(
        load_vals,
        large_ecmp,
        marker='s',
        linewidth=2,
        label='ECMP'
    )

    plt.plot(
        load_vals,
        large_rtt,
        marker='^',
        linewidth=2,
        label='RTT'
    )

    plt.xlabel("Network Load")
    plt.ylabel("Average FCT (s)")

    plt.title(
        f"{benchmark}: Large Flows (>1MB)"
    )

    plt.legend()
    plt.grid(True)

    out_path = (
        graph_dir / "fct /
        f"{benchmark}_LARGE_vs_load.png"
    )

    plt.savefig(out_path, bbox_inches='tight')
    plt.close()

    print(f"Saved: {out_path}")

print("\nDone: Small vs Large flow comparison (Conga vs ECMP vs RTT)")


===== private_enterprise =====
Saved: /home/hsd/workspace/ns3-load-balance/results_rtt_fix/analysis_results/graph/private_enterprise_SMALL_vs_load.png
Saved: /home/hsd/workspace/ns3-load-balance/results_rtt_fix/analysis_results/graph/fct/private_enterprise_LARGE_vs_load.png

===== social_media_cloud =====
Saved: /home/hsd/workspace/ns3-load-balance/results_rtt_fix/analysis_results/graph/social_media_cloud_SMALL_vs_load.png
Saved: /home/hsd/workspace/ns3-load-balance/results_rtt_fix/analysis_results/graph/fct/social_media_cloud_LARGE_vs_load.png

===== commercial_cloud =====
Saved: /home/hsd/workspace/ns3-load-balance/results_rtt_fix/analysis_results/graph/commercial_cloud_SMALL_vs_load.png
Saved: /home/hsd/workspace/ns3-load-balance/results_rtt_fix/analysis_results/graph/fct/commercial_cloud_LARGE_vs_load.png

===== university =====
Saved: /home/hsd/workspace/ns3-load-balance/results_rtt_fix/analysis_results/graph/university_SMALL_vs_load.png
Saved: /home/hsd/workspace/ns3-load-balanc

In [4]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import numpy as np

# ------------------ CONFIG ------------------

save_dir = Path(
    "/home/hsd/workspace/ns3-load-balance/results_rtt_fix/analysis_results/"
)

graph_dir = save_dir / "graph"
graph_dir.mkdir(exist_ok=True)

benchmarks = [
    "private_enterprise",
    "social_media_cloud",
    "commercial_cloud",
    "university"
]

loads = range(1, 10)

SMALL = 100 * 1024        # 100 KB
LARGE = 1 * 1024 * 1024  # 1 MB

# ------------------ MAIN ------------------

for benchmark in benchmarks:

    print(f"\n===== {benchmark} =====")

    small_conga, small_ecmp, small_rtt = [], [], []
    large_conga, large_ecmp, large_rtt = [], [], []

    load_vals = []

    for load in loads:

        csv_path = save_dir / f"{benchmark}_load_{load}_full.csv"

        # Create the "graph" directory if it doesn't exist
        _path.parent.mkdir(parents=True, exist_ok=True)

        if not csv_path.exists():
            print(f"Missing: {csv_path}")
            continue

        df = pd.read_csv(csv_path)

        if df.empty:
            continue

        # --------------------------------------------------
        # FLOW SIZE SPLIT
        # --------------------------------------------------

        small_df = df[df['flow_size'] < SMALL]
        large_df = df[df['flow_size'] > LARGE]

        if len(small_df) > 0:
            small_conga.append(small_df['FCT(s)_conga'].mean())
            small_ecmp.append(small_df['FCT(s)_ecmp'].mean())
            small_rtt.append(small_df['FCT(s)_rtt'].mean())
        else:
            small_conga.append(np.nan)
            small_ecmp.append(np.nan)
            small_rtt.append(np.nan)

        if len(large_df) > 0:
            large_conga.append(large_df['FCT(s)_conga'].mean())
            large_ecmp.append(large_df['FCT(s)_ecmp'].mean())
            large_rtt.append(large_df['FCT(s)_rtt'].mean())
        else:
            large_conga.append(np.nan)
            large_ecmp.append(np.nan)
            large_rtt.append(np.nan)

        load_vals.append(load / 10)

        # --------------------------------------------------
        # SIZE MASKS
        # --------------------------------------------------

        small_mask = df['flow_size'] < SMALL
        large_mask = df['flow_size'] > LARGE
        mid_mask = ~(small_mask | large_mask)

        # --------------------------------------------------
        # ONLY RTT COMPARISONS
        # --------------------------------------------------

        pairs = [
            ("conga", "rtt"),
            ("ecmp", "rtt")
        ]

        for a, b in pairs:

            plt.figure(figsize=(7, 7))

            # Small flows
            plt.scatter(
                df.loc[small_mask, f'FCT(s)_{a}'],
                df.loc[small_mask, f'FCT(s)_{b}'],
                alpha=0.5,
                label='Small (<100KB)'
            )

            # Medium flows
            plt.scatter(
                df.loc[mid_mask, f'FCT(s)_{a}'],
                df.loc[mid_mask, f'FCT(s)_{b}'],
                alpha=0.5,
                label='Medium (100KB-1MB)'
            )

            # Large flows
            plt.scatter(
                df.loc[large_mask, f'FCT(s)_{a}'],
                df.loc[large_mask, f'FCT(s)_{b}'],
                alpha=0.5,
                label='Large (>1MB)'
            )

            max_val = max(
                df[f'FCT(s)_{a}'].max(),
                df[f'FCT(s)_{b}'].max()
            )

            plt.plot(
                [0, max_val],
                [0, max_val],
                linestyle='--',
                color='black',
                label='y=x'
            )

            plt.xlabel(f"{a.upper()} FCT (s)")
            plt.ylabel(f"{b.upper()} FCT (s)")

            plt.title(
                f"{benchmark} Load 0.{load}\n"
                f"{b.upper()} vs {a.upper()} (Flow Size Colored)"
            )

            plt.legend()
            plt.grid(True)

            out_file = (
                graph_dir /
                f"{benchmark}_load_{load}_{a}_{b}_colored_scatter.png"
            )

            plt.savefig(
                out_file,
                bbox_inches='tight',
                dpi=300
            )

            plt.close()

            print(f"Saved: {out_file}")

print("\nDone: RTT vs Conga and RTT vs ECMP colored scatter plots generated.")


===== private_enterprise =====
Saved: /home/hsd/workspace/ns3-load-balance/results_rtt_fix/analysis_results/graph/private_enterprise_load_1_conga_rtt_colored_scatter.png
Saved: /home/hsd/workspace/ns3-load-balance/results_rtt_fix/analysis_results/graph/private_enterprise_load_1_ecmp_rtt_colored_scatter.png
Saved: /home/hsd/workspace/ns3-load-balance/results_rtt_fix/analysis_results/graph/private_enterprise_load_2_conga_rtt_colored_scatter.png
Saved: /home/hsd/workspace/ns3-load-balance/results_rtt_fix/analysis_results/graph/private_enterprise_load_2_ecmp_rtt_colored_scatter.png
Saved: /home/hsd/workspace/ns3-load-balance/results_rtt_fix/analysis_results/graph/private_enterprise_load_3_conga_rtt_colored_scatter.png
Saved: /home/hsd/workspace/ns3-load-balance/results_rtt_fix/analysis_results/graph/private_enterprise_load_3_ecmp_rtt_colored_scatter.png
Saved: /home/hsd/workspace/ns3-load-balance/results_rtt_fix/analysis_results/graph/private_enterprise_load_4_conga_rtt_colored_scatter.pn

In [5]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import numpy as np

# ------------------ CONFIG ------------------

save_dir = Path(
    "/home/hsd/workspace/ns3-load-balance/results_rtt_fix/analysis_results/"
)

graph_dir = save_dir / "graph/fct/normalised"
graph_dir.mkdir(parents=True, exist_ok=True)

benchmarks = [
    "private_enterprise",
    "social_media_cloud",
    "commercial_cloud",
    "university"
]

loads = range(1, 10)

SMALL = 100 * 1024
LARGE = 1 * 1024 * 1024

# =========================================================
# MAIN
# =========================================================

for benchmark in benchmarks:

    print(f"\n===== {benchmark} =====")

    load_vals = []

    # RTT / ECMP
    small_rtt_ecmp = []
    large_rtt_ecmp = []

    # RTT / CONGA
    small_rtt_conga = []
    large_rtt_conga = []

    for load in loads:

        csv_path = save_dir / f"{benchmark}_load_{load}_full.csv"

        # Create the "graph" directory if it doesn't exist
        csv_path.parent.mkdir(parents=True, exist_ok=True)

        if not csv_path.exists():
            continue

        df = pd.read_csv(csv_path)

        if df.empty:
            continue

        # -------------------------------------------------
        # Avoid divide-by-zero
        # -------------------------------------------------

        df = df[
            (df['FCT(s)_ecmp'] > 0) &
            (df['FCT(s)_conga'] > 0)
        ]

        if df.empty:
            continue

        # -------------------------------------------------
        # Normalized FCT
        # -------------------------------------------------

        df['rtt_vs_ecmp'] = (
            df['FCT(s)_rtt'] /
            df['FCT(s)_ecmp']
        )

        df['rtt_vs_conga'] = (
            df['FCT(s)_rtt'] /
            df['FCT(s)_conga']
        )

        # -------------------------------------------------
        # Flow size split
        # -------------------------------------------------

        small_df = df[df['flow_size'] < SMALL]
        large_df = df[df['flow_size'] > LARGE]

        # SMALL FLOWS

        if not small_df.empty:

            small_rtt_ecmp.append(
                small_df['rtt_vs_ecmp'].mean()
            )

            small_rtt_conga.append(
                small_df['rtt_vs_conga'].mean()
            )

        else:

            small_rtt_ecmp.append(np.nan)
            small_rtt_conga.append(np.nan)

        # LARGE FLOWS

        if not large_df.empty:

            large_rtt_ecmp.append(
                large_df['rtt_vs_ecmp'].mean()
            )

            large_rtt_conga.append(
                large_df['rtt_vs_conga'].mean()
            )

        else:

            large_rtt_ecmp.append(np.nan)
            large_rtt_conga.append(np.nan)

        load_vals.append(load / 10)

    # =====================================================
    # SMALL FLOWS
    # =====================================================

    plt.figure(figsize=(8, 6), dpi=120)

    plt.plot(
        load_vals,
        small_rtt_ecmp,
        marker='o',
        linewidth=2,
        label='RTT / ECMP'
    )

    plt.plot(
        load_vals,
        small_rtt_conga,
        marker='s',
        linewidth=2,
        label='RTT / Conga'
    )

    plt.axhline(
        y=1,
        linestyle='--',
        color='black'
    )

    plt.xlabel("Network Load")
    plt.ylabel("Relative FCT")

    plt.title(
        f"{benchmark}: Small Flows (<100KB)"
    )

    plt.legend()
    plt.grid(True)

    plt.ylim(0, 2)

    out_path = (
        graph_dir /
        f"{benchmark}_SMALL_relative_vs_load.png"
    )

    # Create the "graph" directory if it doesn't exist
    out_path.parent.mkdir(parents=True, exist_ok=True)

    plt.savefig(
        out_path,
        bbox_inches='tight'
    )

    plt.close()

    print(f"Saved: {out_path}")

    # =====================================================
    # LARGE FLOWS
    # =====================================================

    plt.figure(figsize=(8, 6), dpi=120)

    plt.plot(
        load_vals,
        large_rtt_ecmp,
        marker='o',
        linewidth=2,
        label='RTT / ECMP'
    )

    plt.plot(
        load_vals,
        large_rtt_conga,
        marker='s',
        linewidth=2,
        label='RTT / Conga'
    )

    plt.axhline(
        y=1,
        linestyle='--',
        color='black'
    )

    plt.xlabel("Network Load")
    plt.ylabel("Relative FCT")

    plt.title(
        f"{benchmark}: Large Flows (>1MB)"
    )

    plt.legend()
    plt.grid(True)

    plt.ylim(0, 2)

    out_path = (
        graph_dir /
        f"{benchmark}_LARGE_relative_vs_load.png"
    )



    plt.savefig(
        out_path,
        bbox_inches='tight'
    )

    plt.close()

    print(f"Saved: {out_path}")

print("\nDone: RTT normalized FCT graphs generated")


===== private_enterprise =====
Saved: /home/hsd/workspace/ns3-load-balance/results_rtt_fix/analysis_results/graph/fct/normalised/private_enterprise_SMALL_relative_vs_load.png
Saved: /home/hsd/workspace/ns3-load-balance/results_rtt_fix/analysis_results/graph/fct/normalised/private_enterprise_LARGE_relative_vs_load.png

===== social_media_cloud =====
Saved: /home/hsd/workspace/ns3-load-balance/results_rtt_fix/analysis_results/graph/fct/normalised/social_media_cloud_SMALL_relative_vs_load.png
Saved: /home/hsd/workspace/ns3-load-balance/results_rtt_fix/analysis_results/graph/fct/normalised/social_media_cloud_LARGE_relative_vs_load.png

===== commercial_cloud =====
Saved: /home/hsd/workspace/ns3-load-balance/results_rtt_fix/analysis_results/graph/fct/normalised/commercial_cloud_SMALL_relative_vs_load.png
Saved: /home/hsd/workspace/ns3-load-balance/results_rtt_fix/analysis_results/graph/fct/normalised/commercial_cloud_LARGE_relative_vs_load.png

===== university =====
Saved: /home/hsd/works